# Ch.6 — RNNs / LSTMs / GRUs
**Track:** ML from Scratch · Synthetic monthly housing price index (time-series forecasting)

## Core Idea

Dense networks and CNNs ignore the **order** of inputs. Sequential data has temporal structure:
last month's price informs this month's price.

```
Dense (Ch.2): [p1, p2, ..., p12] treated as an unordered feature vector
RNN: processes p1 → p2 → ... → p12 one at a time, carrying h_t forward
LSTM: adds a gated cell state c_t — a protected memory highway
```

**The key equations:**
- RNN: $\mathbf{h}_t = \tanh(\mathbf{W}_{hh}\mathbf{h}_{t-1} + \mathbf{W}_{xh}\mathbf{x}_t + \mathbf{b}_h)$
- LSTM: $\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$ (cell state update)
- GRU: $\mathbf{h}_t = (1{-}\mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$

## Running Example

The real estate platform wants a **monthly housing price index forecaster**:
given the last 12 months of a district's median house value, predict next month's value.

We generate a **synthetic 120-month price series** (10 years):
linear trend + annual seasonal cycle + Gaussian noise — exactly what an LSTM can decompose.

In [ ]:
def make_price_series(n_months=120, seed=42):
    """
    TODO #1: Implement `make_price_series()`.

    Steps:
    1. Process data
    2. Synthetic monthly housing price index
    3. Compute `prices` using `arange()`
    4. Call `mean()` to produce the result

    Hint:
    rng = np.random.default_rng(???)
    t = np.arange(???)
    noise = rng.normal(???)
    months = np.arange(???)

    Returns: (2.0 + trend + seasonal + noise).asty...
    """
    raise NotImplementedError("TODO: implement make_price_series()")

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Plot results -- call `subplots()`
# 2. Plot results -- call `plot()`
# 3. Compute `trend_line` using `sin()`
# 4. Plot results -- call `plot()`
# 5. Plot results -- call `suptitle()`
#
# Hint:
#    axes = plt.subplots(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(months, prices, color='steelblue', linewidth=1.5)
axes[0].set_title('Synthetic Monthly Housing Price Index')
axes[0].set_xlabel('Month'); axes[0].set_ylabel('Median house value ($100k)')
axes[0].grid(alpha=0.3)

# Decompose: trend + seasonal + residual
trend_line = 2.0 + 0.005 * months
seasonal   = 0.15 * np.sin(2 * np.pi * months / 12)

axes[1].plot(months, prices,   label='Full series',   color='steelblue', alpha=0.5)
axes[1].plot(months, trend_line, label='Trend',       color='coral',     linewidth=2)
axes[1].plot(months, trend_line + seasonal, label='Trend+Seasonal', color='green', linewidth=1.5, linestyle='--')
axes[1].set_title('Series Decomposition')
axes[1].set_xlabel('Month'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Ch.6 Running Example — Price Series the LSTM Must Learn', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Sliding Window Dataset

Convert the 1-D price series into `(X, y)` pairs:
- `X[i]` = months `i` through `i+T-1` (shape `(T, 1)`)
- `y[i]` = month `i+T` (the next value to predict)

**Critical:** never shuffle time-series splits. Test set must be the most recent months.

In [ ]:
def make_windows(series, T=12):
    """
    TODO #3: Implement `make_windows()`.

    Steps:
    1. Define helper function `make_windows()`
    2. Compute `T`
    3. Compute `split`
    4. Compute `mean_val` using `only()`
    5. Compute `X_train`
    6. Call `mean()` to produce the result

    Hint:
    mean_val = X_train.mean(???)
    std_val = X_train.std(???)

    Returns: np.array(X_list, dtype=np.float32), n...
    """
    raise NotImplementedError("TODO: implement make_windows()")

## Manual RNN Forward Pass (NumPy)

$$\mathbf{h}_t = \tanh\!\left(\mathbf{W}_{hh}\mathbf{h}_{t-1} + \mathbf{W}_{xh}\mathbf{x}_t + \mathbf{b}_h\right)$$

The same weight matrices $\mathbf{W}_{hh}$ and $\mathbf{W}_{xh}$ are applied at **every** time step.

In [ ]:
def rnn_forward(X_seq, W_xh, W_hh, b_h, W_hy, b_y):
    """
    TODO #4: Implement `rnn_forward()`.

    Steps:
    1. Define helper function `rnn_forward()`
    2. Compute `rng` using `normal()`
    3. Process data

    Hint:
    h = np.zeros(???)
    h = np.tanh(???)
    rng = np.random.default_rng(???)
    W_xh_demo = rng.normal(???)

    Returns: np.array(hidden_states), y_hat
    """
    raise NotImplementedError("TODO: implement rnn_forward()")

## Vanishing Gradient — Visualised

How far does a gradient signal from the output reach through time?
Each step multiplies by $\mathbf{W}_{hh}^\top$ and the Jacobian of $\tanh$.

$$\frac{\partial \mathbf{h}_T}{\partial \mathbf{h}_0} = \prod_{t=1}^{T} \mathbf{W}_{hh}^\top \cdot \mathrm{diag}(1 - \mathbf{h}_t^2)$$

When the spectral radius of $\mathbf{W}_{hh} < 1$, this product decays to zero exponentially.

In [ ]:
def gradient_norm_through_time(W_hh, T=50, n_trials=100, seed=1):
    """
    TODO #5: Implement `gradient_norm_through_time()`.

    Steps:
    1. Define helper function `gradient_norm_through_time()`
    2. Compute `H_vg` using `default_rng()`
    3. Compute `W_small` using `normal()`
    4. Compute `T_steps`
    5. Plot results -- call `semilogy()`
    6. Process data

    Hint:
    rng = np.random.default_rng(???)
    grad = np.ones(???)
    h = rng.normal(???)
    h = np.tanh(???)

    Returns: np.mean(norms, axis=0)
    """
    raise NotImplementedError("TODO: implement gradient_norm_through_time()")

## LSTM — Training on the Price Series

The LSTM cell state $\mathbf{c}_t$ is updated **additively** (not multiplicatively),
so gradients can flow back 12+ steps without vanishing:

$$\mathbf{c}_t = \underbrace{\mathbf{f}_t \odot \mathbf{c}_{t-1}}_{\text{selective forget}} + \underbrace{\mathbf{i}_t \odot \tilde{\mathbf{c}}_t}_{\text{selective write}}$$

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
#
# Hint:
#    # implement using the APIs described above

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    TF_AVAILABLE = True
    print(f"TensorFlow {tf.__version__} available")
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow not installed — Keras training cells will be skipped.")
    print("Install with: pip install tensorflow")

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `set_seed()` to produce the result
# 2. Call `Dense()` to produce the result
# 3. Call `compile()` to produce the result
#
# Hint:
#    lstm_model = keras.Sequential(???)
#    optimizer = keras.optimizers.Adam(???)
#    lstm_model.compile(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
if TF_AVAILABLE:
    tf.random.set_seed(42)

    lstm_model = keras.Sequential([
        layers.Input(shape=(T, 1)),
        layers.LSTM(64, return_sequences=False),
        layers.Dense(32, activation='relu'),
        layers.Dense(1),            # regression — no activation
    ], name='HousePriceForecaster_LSTM')

    lstm_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
        loss='mse', metrics=['mae']
    )
    lstm_model.summary()
else:
    print("Skipped — TensorFlow not available.")

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `EarlyStopping()` to produce the result
# 2. Fit the model -- call `fit()`
# 3. Fit the model -- call `predict()`
# 4. Process data
# 5. Call `sqrt()` to produce the result
#
# Hint:
#    early_stop = keras.callbacks.EarlyStopping(???)
#    history_lstm = lstm_model.fit(???)
#    y_pred_lstm = lstm_model.predict(???)
#    rmse_lstm = np.sqrt(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
if TF_AVAILABLE:
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20, restore_best_weights=True)

    history_lstm = lstm_model.fit(
        X_train, y_train,
        epochs=200, batch_size=16,
        validation_split=0.15,    # last 15% of training data (chronological)
        callbacks=[early_stop],
        verbose=0,
    )

    y_pred_lstm = lstm_model.predict(X_test, verbose=0).ravel()

    # Inverse-transform to original scale
    y_pred_real  = y_pred_lstm * std_val + mean_val
    y_test_real  = y_test      * std_val + mean_val

    rmse_lstm = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
    mae_lstm  = mean_absolute_error(y_test_real, y_pred_real)
    print(f"LSTM  — RMSE: {rmse_lstm:.4f}  MAE: {mae_lstm:.4f}  (units: $100k)")
else:
    print("Skipped — TensorFlow not available.")

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `arange()` to produce the result
# 2. Plot results -- call `subplots()`
# 3. Plot results -- call `plot()`
# 4. Plot results -- call `suptitle()`
#
# Hint:
#    test_months = np.arange(???)
#    axes = plt.subplots(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
if TF_AVAILABLE:
    # Plot: actual vs predicted on test set
    test_months = np.arange(split + T, len(prices))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(months,               prices,        color='lightgrey',   label='Full series')
    axes[0].plot(test_months,          y_test_real,   color='steelblue',   label='Actual (test)', linewidth=2)
    axes[0].plot(test_months,          y_pred_real,   color='coral',       label='LSTM predicted',
                 linewidth=2, linestyle='--')
    axes[0].axvline(split + T, color='black', linestyle=':', alpha=0.5, label='Train/test split')
    axes[0].set_title('LSTM: Actual vs Predicted'); axes[0].set_xlabel('Month')
    axes[0].set_ylabel('Value ($100k)'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    axes[1].plot(history_lstm.history['loss'],     label='Train loss', color='steelblue')
    axes[1].plot(history_lstm.history['val_loss'], label='Val loss',   color='coral', linestyle='--')
    axes[1].set_title('Loss Curve'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.suptitle(f'LSTM Forecaster — RMSE={rmse_lstm:.4f} MAE={mae_lstm:.4f} ($100k)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — TensorFlow not available.")

## GRU — Drop-in Comparison

The GRU replaces LSTM's three gates with two (reset + update) and eliminates the separate cell state.
Roughly 25% fewer parameters for the same hidden size.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `set_seed()` to produce the result
# 2. Call `Dense()` to produce the result
# 3. Call `compile()` to produce the result
# 4. Fit the model -- call `fit()`
# 5. Fit the model -- call `predict()`
# 6. Call `count_params()` to produce the result
# 7. Process data
#
# Hint:
#    gru_model = keras.Sequential(???)
#    optimizer = keras.optimizers.Adam(???)
#    history_gru = gru_model.fit(???)
#    y_pred_gru = gru_model.predict(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
if TF_AVAILABLE:
    tf.random.set_seed(42)

    gru_model = keras.Sequential([
        layers.Input(shape=(T, 1)),
        layers.GRU(64, return_sequences=False),
        layers.Dense(32, activation='relu'),
        layers.Dense(1),
    ], name='HousePriceForecaster_GRU')

    gru_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
        loss='mse', metrics=['mae']
    )

    history_gru = gru_model.fit(
        X_train, y_train,
        epochs=200, batch_size=16,
        validation_split=0.15,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=20, restore_best_weights=True)],
        verbose=0,
    )

    y_pred_gru  = gru_model.predict(X_test, verbose=0).ravel()
    y_pred_gru_real = y_pred_gru * std_val + mean_val
    rmse_gru = np.sqrt(mean_squared_error(y_test_real, y_pred_gru_real))
    mae_gru  = mean_absolute_error(y_test_real, y_pred_gru_real)

    lstm_params = lstm_model.count_params()
    gru_params  = gru_model.count_params()

    print(f"LSTM — RMSE: {rmse_lstm:.4f}  MAE: {mae_lstm:.4f}  Params: {lstm_params:,}")
    print(f"GRU  — RMSE: {rmse_gru:.4f}  MAE: {mae_gru:.4f}  Params: {gru_params:,}")
    print(f"Parameter reduction: {(1 - gru_params/lstm_params)*100:.0f}%")
else:
    print("Skipped — TensorFlow not available.")

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Plot results -- call `subplots()`
# 2. Plot results -- call `plot()`
# 3. Plot results -- call `suptitle()`
#
# Hint:
#    axes = plt.subplots(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
if TF_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(test_months, y_test_real,     color='black',     label='Actual', linewidth=2)
    axes[0].plot(test_months, y_pred_real,     color='steelblue', label='LSTM',   linewidth=2, linestyle='--')
    axes[0].plot(test_months, y_pred_gru_real, color='coral',     label='GRU',    linewidth=2, linestyle=':')
    axes[0].set_title('LSTM vs GRU — Predictions on Test Set')
    axes[0].set_xlabel('Month'); axes[0].set_ylabel('Value ($100k)')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Learning curve comparison
    axes[1].plot(history_lstm.history['val_loss'], color='steelblue', label=f'LSTM val loss')
    axes[1].plot(history_gru.history['val_loss'],  color='coral',     label=f'GRU val loss', linestyle='--')
    axes[1].set_title('Validation Loss: LSTM vs GRU')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.suptitle('LSTM vs GRU Comparison', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — TensorFlow not available.")

## Hyperparameter Dial — Sequence Length $T$

How does the look-back window size affect prediction quality?
Too short: misses the seasonal cycle. Too long: more gradient propagation, slower training.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `make_windows()` to produce the result
# 2. Fit the model -- call `set_seed()`
# 3. Plot results -- call `subplots()`
# 4. Call `months()` to produce the result
#
# Hint:
#    sig = Xtr.mean(???)
#    m = keras.Sequential(???)
#    preds = m.predict(???)
#    rmse_w = np.sqrt(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
if TF_AVAILABLE:
    results = {}
    for window_T in [3, 6, 12, 24]:
        X_w, y_w  = make_windows(prices, T=window_T)
        split_w   = int(len(X_w) * 0.8)
        Xtr, Xte  = X_w[:split_w], X_w[split_w:]
        ytr, yte  = y_w[:split_w], y_w[split_w:]
        mu, sig   = Xtr.mean(), Xtr.std()
        Xtr = (Xtr - mu) / sig;  Xte = (Xte - mu) / sig
        ytr = (ytr - mu) / sig;  yte = (yte - mu) / sig

        tf.random.set_seed(42)
        m = keras.Sequential([
            layers.Input(shape=(window_T, 1)),
            layers.LSTM(32),
            layers.Dense(1),
        ])
        m.compile(optimizer='adam', loss='mse')
        m.fit(Xtr, ytr, epochs=100, batch_size=16,
              validation_split=0.15, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(patience=10,
                                                       restore_best_weights=True)])
        preds = m.predict(Xte, verbose=0).ravel() * sig + mu
        actual = yte * sig + mu
        rmse_w = np.sqrt(mean_squared_error(actual, preds))
        results[window_T] = rmse_w
        print(f"T={window_T:2d} months → RMSE: {rmse_w:.4f}")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar([str(k) for k in results], results.values(), color='steelblue', edgecolor='white')
    ax.set_xlabel('Sequence length T (months)')
    ax.set_ylabel('Test RMSE ($100k)')
    ax.set_title('Effect of Look-back Window on Forecast Accuracy')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    best_T = min(results, key=results.get)
    print(f"\nBest window: T={best_T} months (RMSE={results[best_T]:.4f})")
    print("T=12 captures the full seasonal cycle — matches the 12-month sin period.")
else:
    print("Skipped — TensorFlow not available.")

## What Can Go Wrong — Demo: Shuffling Time-Series Data

Shuffling a time-series dataset before splitting leaks future prices into training.
Compare the honest (chronological) score vs the inflated (shuffled) score.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `make_windows()` to produce the result
# 2. Call `split()` to produce the result
# 3. Fit the model -- call `set_seed()`
# 4. Process data
#
# Hint:
#    idx = np.random.default_rng(???)
#    sig_s = Xtr_s.mean(???)
#    m_bad = keras.Sequential(???)
#    pred_bad = m_bad.predict(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
if TF_AVAILABLE:
    X_all, y_all = make_windows(prices, T=12)

    # WRONG: shuffle before split (leaks future into train)
    idx         = np.random.default_rng(99).permutation(len(X_all))
    X_shuf, y_shuf = X_all[idx], y_all[idx]
    sp          = int(len(X_shuf) * 0.8)
    Xtr_s, Xte_s = X_shuf[:sp], X_shuf[sp:]
    ytr_s, yte_s = y_shuf[:sp], y_shuf[sp:]
    mu_s, sig_s  = Xtr_s.mean(), Xtr_s.std()
    Xtr_s = (Xtr_s - mu_s)/sig_s;  Xte_s = (Xte_s - mu_s)/sig_s
    ytr_s = (ytr_s - mu_s)/sig_s;  yte_s = (yte_s - mu_s)/sig_s

    tf.random.set_seed(42)
    m_bad = keras.Sequential([
        layers.Input(shape=(12, 1)), layers.LSTM(64), layers.Dense(1)])
    m_bad.compile(optimizer='adam', loss='mse')
    m_bad.fit(Xtr_s, ytr_s, epochs=100, batch_size=16, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
    pred_bad  = m_bad.predict(Xte_s, verbose=0).ravel() * sig_s + mu_s
    rmse_bad  = np.sqrt(mean_squared_error(yte_s * sig_s + mu_s, pred_bad))

    print(f"Chronological split RMSE:  {rmse_lstm:.4f}  (honest)")
    print(f"Shuffled split RMSE:      {rmse_bad:.4f}  (inflated — future leaked into train)")
    print(f"\nThe shuffled model looks {(rmse_lstm/rmse_bad - 1)*100:.0f}% better than it really is.")
else:
    print("Skipped — TensorFlow not available.")

## Exercises

1. **Stacked LSTM:** add a second LSTM layer (`return_sequences=True` on the first). Does it improve RMSE?
2. **Dropout:** add `layers.Dropout(0.2)` between the LSTM and Dense layers. Compare val vs train loss curves — does the gap narrow?
3. **Multi-step forecast:** instead of predicting month T+1, modify `make_windows` to predict the next 3 months (`y.shape = (N, 3)`) and add `return_sequences=True` to get one prediction per step.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Implement the cell logic
#
# Hint:
#    # implement using the APIs described above

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# Exercise 1: Stacked LSTM
# TODO: build a keras.Sequential with two LSTM layers.
# First LSTM must use return_sequences=True.
# Compare test RMSE to the single-layer LSTM above.

# Exercise 2: Dropout regularisation
# TODO: add layers.Dropout(0.2) between LSTM and Dense.
# Plot train loss vs val loss — does the gap shrink?

# Exercise 3: Multi-step forecast
# TODO: modify make_windows to return y of shape (N, 3).
# Use return_sequences=False, Dense(3) as the output layer.
# Report RMSE for each of the 3 future steps separately.